# 서울 전통시장 생필품 가격 분석 (2023~2025)

## 프로젝트 개요

| 항목 | 내용 |
|------|------|
| **목적** | 서울 전통시장 생필품 가격의 시계열 패턴·지역별 격차·외부충격 영향을 정량화하고 시민 체감형 물가 대시보드 제작 |
| **데이터** | 서울 열린데이터 광장 – 생필품 농수축산물 가격 정보 |
| **분석 기간** | 2023년 2월 ~ 2025년 (주 1회 화요일 조사) |
| **분석 범위** | 서울 25개 자치구 × 약 100개 품목 |
| **최종 수정** | 2026-04-28 |

## 분석 흐름
```
[데이터 로드] → [통합 & 정제] → [파생변수 생성] → [이상치 처리] → [분류 체계 구축] → [EDA]
```


## 1. 라이브러리 Import

In [26]:
import re
import os
import gc
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# ── 시각화 설정 ────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'AppleGothic'   # Windows: 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── pandas 출력 옵션 ───────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 2. 데이터 로드 & 통합

### 2-1. 원본 파일 로드

> **파일별 특이사항**
> - `df23` : 칼럼명 한글, `ym` 포맷 `2023-02`
> - `df24` : 칼럼명 한글, `ym` 포맷 `Jan-24` (영문 약자)
> - `df25` : 칼럼명 영문(원본 그대로), `ym` 포맷 `2025-01`


In [ ]:
PATH = '/Users/hyun/Documents/project/seoul-market-price-analysis/data/raw/'

df25 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2025년이후).csv',
                   header=1, low_memory=False)
df24 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2024년).csv',
                   encoding='cp949', low_memory=False)
df23 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2023년).csv',
                   encoding='cp949', low_memory=False)

print(f"df25: {df25.shape}  \ndf24: {df24.shape}  \ndf23: {df23.shape}")

df25: (144684, 19)  
df24: (98053, 14)  
df23: (76559, 14)


### 2-2. 칼럼명 통일 (한글 → 영문)

In [28]:
# df23·df24 한글 칼럼명 → df25 영문 칼럼명 기준으로 통일
col_mapping = {
    '일련번호':                         'sn',
    '시장/마트 번호':                   'mkplc_mart_no',
    '시장/마트 이름':                   'mkplc_mart_nm',
    '품목 번호':                        'prdlst_no',
    '품목 이름':                        'prdlst_nm',
    '실판매규격':                       'real_sle_stndrd',
    '가격(원)':                         'pc',
    '년도-월':                          'ym',
    '비고':                             'rmrk',
    '시장유형 구분(시장/마트) 코드':    'mkplc_type_cd',
    '시장유형 구분(시장/마트) 이름':    'mkplc_type_nm',
    '자치구 코드':                      'atdrc_cd',
    '자치구 이름':                      'atdrc',
    '점검일자':                         'chck_ymd',
}

df23.rename(columns=col_mapping, inplace=True)
df24.rename(columns=col_mapping, inplace=True)

### 2-3. `ym` 포맷 통일 및 2023-01 제거

In [29]:
# df23 : '2023-02' 형식 → 그대로 파싱
df23['ym'] = pd.to_datetime(df23['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')

# df24 : 'Jan-24' 형식 → 변환
df24['ym'] = pd.to_datetime(df24['ym'], format='%b-%y', errors='coerce').dt.strftime('%Y-%m')

# df25 : '2025-01' 형식 → 그대로 파싱
df25['ym'] = pd.to_datetime(df25['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')

# 2023-01 제거 : 데이터가 98건으로 매우 적고 2023-02 데이터도 없어 시계열 연속성 확보 불가
df23 = df23[df23['ym'] != '2023-01']

# mkplc_type_cd 타입 통일 (24년 float → Int64)
for df in [df23, df24, df25]:
    df['mkplc_type_cd'] = df['mkplc_type_cd'].astype('Int64')

print("ym 포맷 통일 완료")
print(f"  df23 기간: {df23['ym'].min()} ~ {df23['ym'].max()}")
print(f"  df24 기간: {df24['ym'].min()} ~ {df24['ym'].max()}")
print(f"  df25 기간: {df25['ym'].min()} ~ {df25['ym'].max()}")

ym 포맷 통일 완료
  df23 기간: 2023-03 ~ 2023-12
  df24 기간: 2024-01 ~ 2024-12
  df25 기간: 2025-01 ~ 2026-04


### 2-4. 데이터 통합 (concat)

In [30]:
df_total = pd.concat([df23, df24, df25], axis=0, ignore_index=True)

print(f"통합 후 전체 데이터: {df_total.shape[0]:,}행 × {df_total.shape[1]}열")
del df23, df24, df25   # 메모리 절약
gc.collect()

통합 후 전체 데이터: 319,198행 × 19열


0

## 3. 전처리 Part 1 – 기본 정제

| 처리 항목 | 방법 | 사유 |
|-----------|------|------|
| 전통시장 필터링 | `mkplc_type_nm == '전통시장'` 유지 | 2023년 3월 이후 대형마트 데이터 제외 기준에 맞춰 전 기간 통일 |
| `chck_ymd` 타입 변환 | `pd.to_datetime` | 시계열 파생변수 생성 전처리 |
| 시계열 정렬 | `sort_values('chck_ymd')` | 연속성 분석 정확도 확보 |


In [31]:
# 전통시장만 유지
df_total = df_total[df_total['mkplc_type_nm'] == '전통시장'].copy()

# chck_ymd datetime 변환 + 정렬
df_total['chck_ymd'] = pd.to_datetime(df_total['chck_ymd'])
df_total = df_total.sort_values('chck_ymd').reset_index(drop=True)

print(f"전통시장 필터링 후: {df_total.shape[0]:,}행")

전통시장 필터링 후: 300,391행


## 4. 전처리 Part 2 – 파생변수 생성

### 4-1. 시계열 파생변수 (반기 / 분기 / 계절)


In [32]:
def get_season(month):
    if month in [3, 4, 5]:   return '봄'
    elif month in [6, 7, 8]: return '여름'
    elif month in [9, 10, 11]:return '가을'
    else:                     return '겨울'

df_total['반기'] = df_total['chck_ymd'].dt.month.apply(lambda m: '상반기' if m <= 6 else '하반기')
df_total['분기'] = df_total['chck_ymd'].dt.quarter
df_total['계절'] = df_total['chck_ymd'].dt.month.apply(get_season)

print("시계열 파생변수 생성 완료:", ['반기', '분기', '계절'])

시계열 파생변수 생성 완료: ['반기', '분기', '계절']


### 4-2. 품목명 파싱 (`prdlst_nm` → `new_std_nm`, `variety`, `spec`)

원본 품목명 예시: `쌀(이천쌀) 20kg`, `고등어(생물) 1마리`, `사과(부사) 3개`

| 파생변수 | 추출 방법 | 예시 결과 |
|----------|-----------|-----------|
| `new_std_nm` | 괄호·숫자·단위 제거 | `쌀` |
| `variety` | 괄호 안 내용 추출 | `이천쌀` |
| `spec` | 숫자+단위 패턴 추출 | `20kg` |


In [33]:
def parse_item_details(text):
    """prdlst_nm → (핵심품목명, 품종, 규격) 분해"""
    if pd.isna(text):
        return '기타', '기본', '규격없음'

    text = str(text)

    # 품종/산지: 괄호 안 텍스트
    v_match = re.search(r'\((.*?)\)', text)
    variety = v_match.group(1).strip() if v_match else '기본'

    # 규격: 숫자 + 단위
    s_match = re.search(r'(\d+(?:\.\d+)?\s*[a-zA-Z가-힣]+)', text)
    spec = s_match.group(1).strip() if s_match else '규격없음'

    # 핵심 품목명: 괄호·숫자·단위·특수문자 제거
    core = re.sub(r'\(.*?\)', '', text)
    core = re.sub(r'\d+(?:\.\d+)?\s*[a-zA-Z가-힣]*', '', core)
    core = re.sub(r'[.,/]', '', core).strip()
    core = ' '.join(core.split())

    return core, variety, spec

parsed = df_total['prdlst_nm'].apply(parse_item_details)
df_total['new_std_nm'] = [x[0] for x in parsed]
df_total['variety']    = [x[1] for x in parsed]
df_total['spec']       = [x[2] for x in parsed]

print(f"품목명 파싱 완료 → 고유 품목 수: {df_total['new_std_nm'].nunique()}개")
print(df_total[['prdlst_nm', 'new_std_nm', 'variety', 'spec']].drop_duplicates().head(8).to_string(index=False))

품목명 파싱 완료 → 고유 품목 수: 125개
  prdlst_nm new_std_nm variety spec
     오렌지 1개        오렌지      기본   1개
  파프리카 100g       파프리카      기본 100g
  파프리카 200g       파프리카      기본 200g
     대파 1kg         대파      기본  1kg
   콩나물 500g        콩나물      기본 500g
      콩 1kg          콩      기본  1kg
콜라 1.8L(1병)         콜라      1병 1.8L
콜라 1.5L(1병)         콜라      1병 1.5L


### 4-2-1 품목코드 맵핑 오류 처리

In [ ]:
# Step 1. prdlst_no 기준 최빈 품목명(new_std_nm) 계산
mode_map = (
    df_total
    .dropna(subset=['new_std_nm'])
    .groupby('prdlst_no')['new_std_nm']
    .agg(lambda x: x.value_counts().index[0])  # 최빈값
    .to_dict()
)

# Step 2. NaN → 최빈값으로 채우기
null_before = df_total['new_std_nm'].isna().sum()
df_total['new_std_nm'] = df_total.apply(
    lambda row: mode_map.get(row['prdlst_no'], row['new_std_nm'])
    if pd.isna(row['new_std_nm']) else row['new_std_nm'],
    axis=1
)
null_after = df_total['new_std_nm'].isna().sum()
print(f"결측치 처리: {null_before:,}건 → {null_after:,}건")

# Step 3. prdlst_no별 정답 품목(최빈값) 지정 및 비율 계산
THRESHOLD = 0.05  # 5% 미만이면 오입력으로 판단

total_counts  = df_total.groupby('prdlst_no')['new_std_nm'].transform('count')
mode_per_code = df_total.groupby('prdlst_no')['new_std_nm'].transform(
    lambda x: x.value_counts().index[0]
)
item_counts   = df_total.groupby(['prdlst_no', 'new_std_nm'])['new_std_nm'].transform('count')

df_total['_ratio']     = item_counts / total_counts
df_total['_mode_item'] = mode_per_code
df_total['_is_wrong']  = (
    (df_total['new_std_nm'] != df_total['_mode_item']) &
    (df_total['_ratio'] < THRESHOLD)
)

# Step 4. drop 전 목록 출력 (육안 확인용)
wrong_summary = (
    df_total[df_total['_is_wrong']]
    .groupby(['prdlst_no', '_mode_item', 'new_std_nm'])
    .size()
    .reset_index(name='건수')
    .rename(columns={'_mode_item': '정답품목', 'new_std_nm': '오입력품목'})
    .sort_values(['prdlst_no', '건수'], ascending=[True, False])
)

print(f"\n⚠️  drop 예정 목록 ({len(wrong_summary)}종, {df_total['_is_wrong'].sum():,}건)")
print(wrong_summary.to_string(index=False))

In [ ]:
# Step 5. 확인 후 실제 drop 실행
df_total = df_total[~df_total['_is_wrong']].copy()

# 임시 칼럼 정리
df_total.drop(columns=['_ratio', '_mode_item', '_is_wrong'], inplace=True)

print(f"오입력 제거 후: {len(df_total):,}행")

### 4-3. 가격 보정 (`pc` → `adj_price`)

품목별로 규격이 혼재(20kg vs 10kg, 1마리 vs 1손 등)하므로  
**단일 기준 단위로 환산**하여 `adj_price` 칼럼 생성

| 품목군 | 목표 단위 | 예시 환산 |
|--------|-----------|-----------|
| 쌀 | 10kg | 20kg → ÷2 |
| 소·돼지·닭고기, 감자 | 100g | 1kg → ÷10 |
| 고등어·갈치·조기 | 1마리 | 1손(2마리) → ÷2 |
| 포도 | 2kg | 1kg → ×2 |
| 배추·무·상추·고구마 | 원가격 유지 | – |


In [34]:
def calculate_standard_price(row):
    """규격별 가격을 단일 기준 단위 가격으로 보정"""
    item    = row['new_std_nm']
    variety = row['variety']
    spec    = row['spec']
    price   = row['pc']

    # 세부 품목명: '쌀_이천쌀', '고등어_생물' 형태
    detailed_nm = f"{item}_{variety}" if variety != '기본' else item

    # 규격에서 숫자 추출
    num_match = re.search(r'(\d+(?:\.\d+)?)', spec)
    amount = float(num_match.group(1)) if num_match else 1.0

    adj_price = price
    is_valid  = True

    if item == '쌀':
        adj_price = (price / amount) * 10 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    elif item in ['소고기', '돼지고기', '닭고기', '감자']:
        if 'kg' in spec:
            adj_price = (price / (amount * 1000)) * 100
        elif 'g' in spec:
            adj_price = (price / amount) * 100
        else:
            is_valid = False

    elif item in ['고등어', '갈치', '조기']:
        if '손' in spec:
            adj_price = (price / (amount * 2))
        elif '마리' in spec:
            adj_price = (price / amount)
        else:
            is_valid = False

    elif item == '포도':
        adj_price = (price / amount) * 2 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    # 채소류(배추·무·상추·고구마): 규격 그대로 수용
    return detailed_nm, adj_price, is_valid

results = df_total.apply(calculate_standard_price, axis=1)
df_total['detailed_nm'] = [r[0] for r in results]
df_total['adj_price']   = [r[1] for r in results]
df_total['is_valid']    = [r[2] for r in results]

df_final = df_total[df_total['is_valid']].copy()
print(f"가격 보정 완료 → 유효 데이터: {len(df_final):,}행")
print(df_final[df_final['new_std_nm'] == '쌀'][['prdlst_nm', 'detailed_nm', 'pc', 'adj_price']].sample(5).to_string(index=False))

가격 보정 완료 → 유효 데이터: 300,391행
      prdlst_nm detailed_nm     pc  adj_price
 쌀(오대쌀) 20kg 1포       쌀_오대쌀  95800   47900.00
 쌀(이천쌀) 10kg 1포       쌀_이천쌀  52000   52000.00
  쌀(오대쌀) 4kg 1포       쌀_오대쌀  30000   75000.00
 쌀(오대쌀) 20kg 1포       쌀_오대쌀 113000   56500.00
 쌀(오대쌀) 10kg 1포       쌀_오대쌀  53800   53800.00


## 5. 전처리 Part 3 – 이상치 처리

### 5-1. 0원 데이터 제거


In [35]:
zero_df = df_final[df_final['adj_price'] == 0]
print(f"0원 데이터: {len(zero_df):,}건")
print("연도별 분포:")
print(pd.to_datetime(zero_df['chck_ymd']).dt.year.value_counts().sort_index())

df_clean = df_final[df_final['adj_price'] > 0].copy()
df_clean.dropna(subset=['adj_price'], inplace=True)
print(f"\n0원 제거 후: {len(df_clean):,}건")

0원 데이터: 43,832건
연도별 분포:
chck_ymd
2023    21226
2024    22574
2025       32
Name: count, dtype: int64

0원 제거 후: 256,559건


### 5-2. 자릿수 오타 감지 및 자동 보정

**판단 기준**: 동일 품목 중앙값 대비 비율이 0.1배 / 10배 수준이면 자릿수 입력 오류로 판단하여 역방향 보정


In [36]:
def auto_correct_price_typos(df):
    """중앙값 대비 비율 기반 자릿수 오타 자동 보정"""
    df_c = df.copy()
    median_dict = df_c.groupby('detailed_nm')['adj_price'].median().to_dict()

    def correct_logic(row):
        price  = row['adj_price']
        median = median_dict.get(row['detailed_nm'], price)
        if median == 0 or pd.isna(price):
            return price, '유지(계산불가)'
        ratio = price / median

        if 0.05 <= ratio <= 0.15:   return price * 10,  '×10 보정'
        elif 0.005 <= ratio <= 0.02: return price * 100, '×100 보정'
        elif 5.0 <= ratio <= 15.0:  return price / 10,  '÷10 보정'
        elif 50.0 <= ratio <= 150.0: return price / 100, '÷100 보정'
        else:                        return price, '유지'

    results = df_c.apply(correct_logic, axis=1)
    df_c['final_price']      = [r[0] for r in results]
    df_c['correction_type']  = [r[1] for r in results]
    return df_c

df_corrected = auto_correct_price_typos(df_clean)

print("자릿수 오타 보정 결과:")
print(df_corrected['correction_type'].value_counts())
corrected = df_corrected[~df_corrected['correction_type'].isin(['유지', '유지(계산불가)'])]
if not corrected.empty:
    print(f"\n실제 보정된 데이터: {len(corrected):,}건")
    print(corrected[['detailed_nm', 'atdrc', 'adj_price', 'final_price', 'correction_type']].head(8).to_string(index=False))

자릿수 오타 보정 결과:
correction_type
유지         254650
÷10 보정       1583
×10 보정        324
×100 보정         1
÷100 보정         1
Name: count, dtype: int64

실제 보정된 데이터: 1,909건
detailed_nm atdrc  adj_price  final_price correction_type
          콩   양천구    1000.00     10000.00          ×10 보정
          콩   양천구     990.00      9900.00          ×10 보정
         수박   양천구    1500.00     15000.00          ×10 보정
         딸기   양천구     600.00      6000.00          ×10 보정
         새우   마포구    4980.00       498.00          ÷10 보정
         생수   양천구    6500.00       650.00          ÷10 보정
         마늘   강남구   15900.00      1590.00          ÷10 보정
        오징어   강남구   27000.00      2700.00          ÷10 보정


### 5-3. 잔존 이상치 제거 (IQR + 100원 미만 컷오프)

자릿수 보정 후에도 남은 이상치를 2단계로 정리

| 단계 | 조건 | 처리 |
|------|------|------|
| 1차 | `final_price < 100` | 전량 제거 |
| 2차 | IQR 경계값 이탈 (단, 중앙값 ±20% 이상 버퍼 적용) | 제거 |


In [ ]:
# def remove_malicious_outliers(df):
#     """IQR + 100원 미만 컷오프로 잔존 이상치 제거"""
#     stats_df = df.groupby('detailed_nm', as_index=False).agg(
#         median=('final_price', 'median'),
#         q1    =('final_price', lambda x: x.quantile(0.25)),
#         q3    =('final_price', lambda x: x.quantile(0.75)),
#     )
#     stats_df['iqr']     = stats_df['q3'] - stats_df['q1']
#     stats_df['lower_b'] = stats_df['q1'] - np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)
#     stats_df['upper_b'] = stats_df['q3'] + np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)

#     df_m = df.merge(stats_df[['detailed_nm', 'lower_b', 'upper_b']], on='detailed_nm', how='left')

#     cond_cut  = df_m['final_price'] < 100
#     cond_iqr  = (df_m['final_price'] < df_m['lower_b']) | (df_m['final_price'] > df_m['upper_b'])
#     is_outlier = cond_cut | cond_iqr

#     df_clean_out = df_m[~is_outlier].drop(columns=['lower_b', 'upper_b'])
#     print(f"이상치 제거 전: {len(df_m):,}건  →  제거 후: {len(df_clean_out):,}건  (제거: {is_outlier.sum():,}건)")
#     return df_clean_out

# df_clean = remove_malicious_outliers(df_corrected)

이상치 제거 전: 256,559건  →  제거 후: 248,465건  (제거: 8,094건)


In [ ]:
# ── 1차: final_price < 100 제거 ──────────────────────────────────────
df_clean = df_corrected[df_corrected['final_price'] >= 100].copy()
print(f"1차 제거 전: {len(df_corrected):,}건  →  제거 후: {len(df_clean):,}건  (제거: {len(df_corrected) - len(df_clean):,}건)")

In [ ]:
# ── 2차: IQR 기준 이상치 제거 ─────────────────────────────────────
def remove_iqr_outliers(df):
    """상세품목명 기준 IQR 이탈 데이터 제거 (중앙값 ±20% 최소 버퍼 적용)"""
    stats_df = df.groupby('detailed_nm', as_index=False).agg(
        median=('final_price', 'median'),
        q1    =('final_price', lambda x: x.quantile(0.25)),
        q3    =('final_price', lambda x: x.quantile(0.75)),
    )
    stats_df['iqr']     = stats_df['q3'] - stats_df['q1']
    stats_df['lower_b'] = stats_df['q1'] - np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)
    stats_df['upper_b'] = stats_df['q3'] + np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)

    df_m = df.merge(stats_df[['detailed_nm', 'lower_b', 'upper_b']], on='detailed_nm', how='left')

    cond_iqr   = (df_m['final_price'] < df_m['lower_b']) | (df_m['final_price'] > df_m['upper_b'])
    df_out     = df_m[~cond_iqr].drop(columns=['lower_b', 'upper_b'])

    print(f"2차 제거 전: {len(df_m):,}건  →  제거 후: {len(df_out):,}건  (제거: {cond_iqr.sum():,}건)")
    return df_out

df_clean = remove_iqr_outliers(df_clean)

## 6. 전처리 Part 4 – 품목 분류 체계 구축

품목명(`new_std_nm`)을 기준으로 **대분류 / 중분류 / 소분류** 매핑


In [38]:
category_dict = {
    # ── 농축수산물 ─────────────────────────────────────────
    '쌀':('농축수산물','농산물','곡류'), '콩':('농축수산물','농산물','곡류'), '팥':('농축수산물','농산물','곡류'),
    '상추':('농축수산물','농산물','채소'), '무':('농축수산물','농산물','채소'), '배추':('농축수산물','농산물','채소'),
    '애호박':('농축수산물','농산물','채소'), '오이':('농축수산물','농산물','채소'), '양파':('농축수산물','농산물','채소'),
    '깻잎':('농축수산물','농산물','채소'), '당근':('농축수산물','농산물','채소'), '파프리카':('농축수산물','농산물','채소'),
    '시금치':('농축수산물','농산물','채소'), '토마토':('농축수산물','농산물','채소'), '깐마늘':('농축수산물','농산물','채소'),
    '대파':('농축수산물','농산물','채소'), '감자':('농축수산물','농산물','채소'), '고구마':('농축수산물','농산물','채소'),
    '콩나물':('농축수산물','농산물','채소'), '버섯':('농축수산물','농산물','채소'), '풋고추':('농축수산물','농산물','채소'),
    '청양고추':('농축수산물','농산물','채소'), '양배추':('농축수산물','농산물','채소'), '꽈리고추':('농축수산물','농산물','채소'),
    '방울토마토':('농축수산물','농산물','채소'), '브로콜리':('농축수산물','농산물','채소'), '부추':('농축수산물','농산물','채소'),
    '쪽파':('농축수산물','농산물','채소'), '가지':('농축수산물','농산물','채소'), '생강':('농축수산물','농산물','채소'),
    '미나리':('농축수산물','농산물','채소'), '도라지':('농축수산물','농산물','채소'), '갓':('농축수산물','농산물','채소'),
    '열무':('농축수산물','농산물','채소'), '호박':('농축수산물','농산물','채소'), '마늘':('농축수산물','농산물','채소'),
    '파':('농축수산물','농산물','채소'), '붉은고추':('농축수산물','농산물','채소'),
    '배':('농축수산물','농산물','과일'), '바나나':('농축수산물','농산물','과일'), '사과':('농축수산물','농산물','과일'),
    '참외':('농축수산물','농산물','과일'), '수박':('농축수산물','농산물','과일'), '오렌지':('농축수산물','농산물','과일'),
    '귤':('농축수산물','농산물','과일'), '단감':('농축수산물','농산물','과일'), '포도':('농축수산물','농산물','과일'),
    '딸기':('농축수산물','농산물','과일'), '골드키위':('농축수산물','농산물','과일'), '복숭아':('농축수산물','농산물','과일'),
    '대추':('농축수산물','농산물','과일'), '밤':('농축수산물','농산물','과일'), '감':('농축수산물','농산물','과일'),
    '소고기':('농축수산물','축산물','정육'), '돼지고기':('농축수산물','축산물','정육'), '닭고기':('농축수산물','축산물','정육'),
    '계란':('농축수산물','축산물','알류'),
    '고등어':('농축수산물','수산물','생선류'), '명태':('농축수산물','수산물','생선류'),
    '갈치':('농축수산물','수산물','생선류'), '조기':('농축수산물','수산물','생선류'),
    '조개':('농축수산물','수산물','해산물'), '오징어':('농축수산물','수산물','해산물'),
    '새우':('농축수산물','수산물','해산물'), '굴':('농축수산물','수산물','해산물'),
    '낙지':('농축수산물','수산물','해산물'), '전복':('농축수산물','수산물','해산물'), '꽃게':('농축수산물','수산물','해산물'),
    '마른멸치':('농축수산물','수산물','건어물/해조류'), '맛김':('농축수산물','수산물','건어물/해조류'),
    # ── 가공식품 ───────────────────────────────────────────
    '설탕':('가공식품','조미료','소스/오일'), '마요네즈':('가공식품','조미료','소스/오일'),
    '식초':('가공식품','조미료','소스/오일'), '식용유':('가공식품','조미료','소스/오일'),
    '케찹':('가공식품','조미료','소스/오일'), '간장':('가공식품','조미료','소스/오일'),
    '참기름':('가공식품','조미료','소스/오일'),
    '새우젓':('가공식품','조미료','젓갈류'), '멸치액젓':('가공식품','조미료','젓갈류'),
    '부침가루':('가공식품','조미료','가루/장류'), '고춧가루':('가공식품','조미료','가루/장류'),
    '된장':('가공식품','조미료','가루/장류'), '고추장':('가공식품','조미료','가루/장류'),
    '밀가루':('가공식품','조미료','가루/장류'), '굵은소금':('가공식품','조미료','가루/장류'), '소금':('가공식품','조미료','가루/장류'),
    '라면':('가공식품','면/빵류','면류'), '컵라면':('가공식품','면/빵류','면류'), '국수':('가공식품','면/빵류','면류'),
    '빵':('가공식품','면/빵류','빵류'),
    '통조림':('가공식품','간편식','간편조리'), '즉석밥':('가공식품','간편식','간편조리'),
    '어묵':('가공식품','간편식','간편조리'), '만두':('가공식품','간편식','간편조리'),
    '김치':('가공식품','간편식','간편조리'), '햄':('가공식품','간편식','가공육'), '소시지':('가공식품','간편식','가공육'),
    '두부':('가공식품','간편식','두부류'),
    '우유':('가공식품','유제품/간식','유제품'), '치즈':('가공식품','유제품/간식','유제품'), '분유':('가공식품','유제품/간식','유제품'),
    '에너지바':('가공식품','유제품/간식','간식류'), '초콜릿':('가공식품','유제품/간식','간식류'), '캔디':('가공식품','유제품/간식','간식류'),
    # ── 음료/주류 ──────────────────────────────────────────
    '사이다':('음료/주류','음료','탄산/생수'), '콜라':('음료/주류','음료','탄산/생수'), '생수':('음료/주류','음료','탄산/생수'),
    '소주':('음료/주류','주류','주류'), '맥주':('음료/주류','주류','주류'),
    # ── 생필품 ────────────────────────────────────────────
    '비누':('생필품','위생용품','바디/헤어'), '샴푸':('생필품','위생용품','바디/헤어'), '바디워시':('생필품','위생용품','바디/헤어'),
    '칫솔':('생필품','위생용품','구강용품'), '치약':('생필품','위생용품','구강용품'),
    '세제':('생필품','생활잡화','세제/세정'), '주방세제':('생필품','생활잡화','세제/세정'), '섬유유연제':('생필품','생활잡화','세제/세정'),
    '위생백':('생필품','생활잡화','주방잡화'), '고무장갑':('생필품','생활잡화','주방잡화'), '일회용컵':('생필품','생활잡화','주방잡화'),
    '휴지':('생필품','위생용품','지류/물티슈'), '물티슈':('생필품','위생용품','지류/물티슈'),
    '기저귀':('생필품','위생용품','위생용품'), '여성용품':('생필품','위생용품','위생용품'),
}

df_category = (pd.DataFrame.from_dict(category_dict, orient='index',
                                       columns=['대분류', '중분류', '소분류'])
                 .reset_index().rename(columns={'index': 'base_nm'}))

# detailed_nm의 '_' 앞부분을 기준으로 조인
df_clean['base_nm'] = df_clean['detailed_nm'].str.split('_').str[0]
df_clean = df_clean.merge(df_category, on='base_nm', how='left')
df_clean[['대분류', '중분류', '소분류']] = df_clean[['대분류', '중분류', '소분류']].fillna('기타')
df_clean.drop(columns=['base_nm'], inplace=True)

print("분류 체계 매핑 완료")
print(df_clean.groupby('대분류').size().rename('건수').to_string())

분류 체계 매핑 완료
대분류
가공식품      71342
기타          247
농축수산물    151691
생필품        9827
음료/주류     15358


## 7. 전처리 Part 5 – 불필요 칼럼 제거 & 칼럼명 한글 통일

### 7-1. 불필요 칼럼 제거


In [39]:
# 분석에 불필요한 칼럼 제거
drop_cols = [
    'mkplc_mart_no',    # 시장번호 – 시장명으로 대체 가능
    'real_sle_stndrd',  # 실제판매규격 – spec 칼럼으로 대체
    'mkplc_type_cd',    # 전통시장 단일 값으로 고정 (필터 후)
    'mkplc_type_nm',    # 동일 사유
    'atdrc_cd',         # 자치구명으로 대체 가능
    'is_valid',         # 내부 처리용 플래그
    'correction_type',  # 내부 처리용 플래그
    'z_score_by_item',  # 이상치 탐지용 임시 칼럼 (있을 경우)
]
drop_cols = [c for c in drop_cols if c in df_clean.columns]
df_clean.drop(columns=drop_cols, inplace=True)
print(f"칼럼 제거 후: {df_clean.shape[1]}개 칼럼")

칼럼 제거 후: 26개 칼럼


### 7-2. 칼럼명 한글 통일

In [40]:
re_col_mapping = {
    'sn':          '일련번호',
    'mkplc_mart_nm': '시장명',
    'prdlst_no':   '품목번호',
    'prdlst_nm':   '품목명',
    'pc':          '가격',
    'ym':          '연월',
    'rmrk':        '비고',
    'atdrc':       '자치구',
    'chck_ymd':    '점검일자',
    'prdlst_cd':   '품목코드',
    'prdlst_std_nm': '품목표준명',
    'vrty_nm':     '품종명',
    'unit':        '단위',
    'qty_nm':      '수량명',
    'new_std_nm':  '대표품목명',
    'detailed_nm': '상세품목명',
    'adj_price':   '보정가격',
    'final_price': '최종가격',
}
# 실제 존재하는 칼럼만 이름 변경
rename_map = {k: v for k, v in re_col_mapping.items() if k in df_clean.columns}
df_clean.rename(columns=rename_map, inplace=True)

print("칼럼명 한글 통일 완료")
print("최종 칼럼 목록:", df_clean.columns.tolist())

칼럼명 한글 통일 완료
최종 칼럼 목록: ['일련번호', '시장명', '품목번호', '품목명', '가격', '연월', '비고', '자치구', '점검일자', '품목코드', '품목표준명', '품종명', '단위', '수량명', '반기', '분기', '계절', '대표품목명', 'variety', 'spec', '상세품목명', '보정가격', '최종가격', '대분류', '중분류', '소분류']


## 8. 전처리 결과 요약

### 8-1. 데이터 감소 현황


In [ ]:
print("="*55)
print("  전처리 단계별 데이터 감소 현황")
print("="*55)
print(f"  원본 (concat 후)         : {len(df_total):>8,} 행")
print(f"  전통시장 필터링 후       : {df_total[df_total['mkplc_type_nm']=='전통시장'].shape[0] if 'mkplc_type_nm' in df_total.columns else 'N/A':>8} 행")
print(f"  가격 보정 유효 데이터    : {len(df_final):>8,} 행")
print(f"  0원 제거 후              : {len(df_clean):>8,} 행")
print(f"  최종 분석 데이터         : {len(df_clean):>8,} 행")
print("="*55)

  전처리 단계별 데이터 감소 현황
  원본 (concat 후)         :  300,391 행
  전통시장 필터링 후       :   300391 행
  가격 보정 유효 데이터    :  300,391 행
  0원 제거 후              :  248,465 행
  최종 분석 데이터         :  248,465 행


### 8-2. 최종 데이터셋 기본 정보

In [42]:
print(f"분석 기간 : {df_clean['점검일자'].min().strftime('%Y-%m-%d')} ~ {df_clean['점검일자'].max().strftime('%Y-%m-%d')}")
print(f"자치구 수 : {df_clean['자치구'].nunique()}개")
print(f"대표 품목 : {df_clean['대표품목명'].nunique()}개")
print(f"상세 품목 : {df_clean['상세품목명'].nunique()}개")
print()
print("[대분류별 데이터 수]")
print(df_clean['대분류'].value_counts().to_string())
print()
print("[결측치 현황]")
null_summary = df_clean.isnull().sum()
print(null_summary[null_summary > 0].to_string() if null_summary.sum() > 0 else "결측치 없음")

분석 기간 : 2023-03-27 ~ 2026-04-13
자치구 수 : 25개
대표 품목 : 125개
상세 품목 : 166개

[대분류별 데이터 수]
대분류
농축수산물    151691
가공식품      71342
음료/주류     15358
생필품        9827
기타          247

[결측치 현황]
시장명         207
품목명         247
비고       220098
품목코드     125045
품목표준명    125045
품종명      125045
단위       145484
수량명      166790


---
## 다음 단계: EDA

```
> 이 노트북에서 생성된 df_clean 을 사용
> 02_eda.ipynb 에서 이어서 진행
```
